# Import Packages (baseline model)

In [ ]:
# univariate multi-step CNN-LSTM for energy usage forecasting
import time

# Start counting notebook running time
time_start = time.time()

import numpy as np
import tensorflow as tf
# Set all random seeds for the program (Python, NumPy, and TensorFlow)
tf.keras.utils.set_random_seed(1)

import pandas as pd
from  IPython.display import display, Image
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

from keras.models import Sequential
from keras.layers import Dense, Flatten, LSTM, RepeatVector, TimeDistributed
from keras.layers import Conv1D, MaxPooling1D
from keras.utils import plot_model
from keras.callbacks import EarlyStopping

In [ ]:
print(tf.__version__)

# Import Data
### Import TDC dataset and parse time stamp column to index

In [ ]:
%%time
df = pd.read_csv(
    '/mnt/c/Users/xin37/github/CNN-LSTM-model-for-energy-usage-forecasting-1/data/TDC2_processed.csv', 
    header=0, 
    parse_dates=['Timestamp'], 
    index_col=['Timestamp']
    ) 
print(df.shape)
display(df.head())

In [4]:
df.index = pd.to_datetime(df.index)
df.index = df.index.tz_localize(None)

# Visualize sample data

In [ ]:
%%time
fig = plt.figure(figsize=(10, 5));
plt.plot(df.index, df['T_Return'], color='blue', linewidth=0.5)
plt.xlabel('Time (date)')
plt.ylabel('Return Air Temperature (°C)')
plt.title('HVAC Return Air Temperature')
plt.show()

# Split a dataset into train/test sets

In [6]:
n_input = 16
test_size = 2 * 60 * 24 * 7  

nwt = len(df) - test_size - n_input

nwp = test_size 

# split dataset into train and test sets
def split_dataset(data, n_output):
    # slice dataset as multiples of output sequence 
    data = data.tail(n_output*nwt) # 
    # split into train and test set
    train = np.array(data.iloc[:-n_output*nwp])
    test =  np.array(data.iloc[-n_output*nwp:-n_output] )
    return train, test

In [ ]:
train, test = split_dataset(df, n_output=7)
print(train.shape)
print(test.shape)

#  Format the data in the overlapping moving window format
We need to restrucutre the time-series data in order for training and making predictions like a supervised machine learning model. For every n_input sequence of data, the n_ouput sequence will be target values. We will also increment the sequences by 1 each time to utilise the so called overlapping window format.

In [8]:
# convert data into input and output steps
def transform_data(train, n_input, n_output):
    X, y = [], []
    # loop over the data by 1 time step
    for i in range(len(train)):
        # define the index range of the input 
        in_end = i + n_input
        out_end = in_end + n_output
        if out_end <= len(train):
            x_input = train[i:in_end, 0] # only one feature/column is selected 
            x_input = x_input.reshape((len(x_input), 1))
            X.append(x_input)
            y.append(train[in_end:out_end, 0])
    return np.array(X), np.array(y)


# Define function for building model
In this model architecture, a one-dimensional convolutional neural network (1D CNN) is used to read and encode the input sequence. An LSTM network is then used as a decoder to make ode-step prediction for each value in the output sequence. 

In [9]:
# build and train the model
def build_model(train, n_input, n_output):
    # transform data
    X_train, y_train = transform_data(train, n_input, n_output)
    # specify model parameters
    verbose, epochs, batch_size = 0, 100, 128
    n_features = X_train.shape[2]
    # reshape output into [samples, timesteps, features]
    y_train = y_train.reshape((y_train.shape[0], y_train.shape[1], 1))
    # define model
    model = Sequential()
    #-------CNN-------
    model.add(Conv1D(32, 3, activation='relu', input_shape=(n_input, n_features)))
    model.add(Conv1D(32, 3, activation='relu'))
    model.add(MaxPooling1D())
    model.add(Flatten())
    #-------CNN-------
    #-------LSTM-------
    model.add(RepeatVector(n_output))
    model.add(LSTM(200, activation='relu', return_sequences=True))
    #-------LSTM-------
    model.add(TimeDistributed(Dense(100, activation='relu')))
    model.add(TimeDistributed(Dense(1)))
    model.compile(loss='mse', optimizer='adam')
    print(model.summary())
    plot_model(model=model, to_file ='model_CNN-LSTM_univariate_multistep_output.png', show_shapes=True, show_layer_names=True)
    # fit network
    # --- CHANGED: Save the training process to 'history' ---
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=1, validation_split=0.2)
    return model, history

# Define function for evaluating model

In [10]:
from tqdm import tqdm
import numpy as np

def evaluate_model(train, test, n_input, n_output):
    model, history = build_model(train, n_input, n_output)

    input_history = list(train[:, 1])
    predictions = []

    n_steps = len(test) // n_output

    for i in tqdm(range(n_steps), desc="Walk-forward prediction"):
        input_x = np.array(input_history[-n_input:], dtype=np.float32)
        input_x = input_x.reshape((1, n_input, 1))

        yhat = model.predict(input_x, verbose=0)

        yhat_sequence = yhat[0]
        predictions.append(yhat_sequence)

        input_history.extend(test[i*n_output:(i+1)*n_output, 1])

    predictions = np.array(predictions)

    return predictions, history

# Train model and make predictions

In [ ]:
%%time
n_output = 1
n_input = 16

# split into train and test
train, test = split_dataset(df, n_output)

# evaluate model and get scores
predictions, history = evaluate_model(train, test, n_input, n_output)
print('number of weeks predicted: ', len(predictions))

# Note: The baseline model doesn't use validation data, so we only plot 'loss'
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train Loss (MSE)', color='blue', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)', color='orange', linewidth=2)
plt.title('Baseline Model: Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

# Visualize model architecture

In [ ]:
Image('model_CNN-LSTM_univariate_multistep_output.png')

# Compare actual and predicted values

In [ ]:
from sklearn.preprocessing import MinMaxScaler

actual = test[:, 1]
predictions_flat = predictions.reshape(-1)

min_len = min(len(actual), len(predictions_flat))
final_actual = actual[:min_len]
final_pred = predictions_flat[:min_len]

rmse = np.sqrt(np.mean((final_actual - final_pred)**2))
mae = np.mean(np.abs(final_actual - final_pred))
r2 = 1 - np.sum((final_actual - final_pred)**2) / np.sum((final_actual - np.mean(final_actual))**2)

print("\n--- BASELINE FINAL RESULTS ---")
print(f"RMSE: {rmse:.4f} °C")
print(f"MAE:  {mae:.4f} °C")
print(f"R²:   {r2:.4f}")

plt.figure(figsize=(20, 8))
plt.plot(final_actual, label='Actual Temperature', color='blue', linewidth=1.5)
plt.plot(final_pred, label='Predicted Temperature', color='orange', alpha=0.8, linewidth=1.5)
plt.title('Final Results: Actual vs Predicted T_Return Temperature')
plt.ylabel('Temperature (°C)')
plt.xlabel('Time Steps (Hours)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# RMSE for all sequences

In [ ]:
def return_rmse(actual, predicted):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    print("The root mean squared error is {:.2f}.".format(rmse))
    
return_rmse(actual, predictions_flat)

In [ ]:
time_end = time.time()
print("Notebook run time: {:.0f} seconds".format(time_end - time_start))